# What is Hugging Face?

**Hugging Face** is an **AI development platform focused on Transformer-based models** that provides everything needed to build, train, share, and run modern language models in a standardized way.

At its core, Hugging Face does not create new AI architectures. Instead, it implements and packages Transformer-based models (such as BERT, GPT, T5, and LLaMA-style models) into reusable software components, along with the data handling, training pipelines, and deployment tools required to use them in real-world systems.

In practical terms, Hugging Face acts as the bridge between Transformer research and production-grade AI applications. It turns complex academic models into reliable, reusable, and scalable building blocks that engineers can confidently use.

### Why Hugging Face exists

Transformer models are powerful but difficult to work with directly because they require:

  - correct architectural implementations

  - specialized tokenization and attention handling

  - efficient training loops

  - careful memory and performance optimizations

Hugging Face solves this by offering **standardized libraries** and shared infrastructure that eliminate the need to re-implement these complex components for every project.

**Hugging Face** components are Transformer Library, Model Hub, Dataset Library, Tokenizers, Pipelines, Trainer API, PEFT/LoRA Integration, Spaces, Metrics and Evaluations. We will talk about each component one by one.

# Pipeline
The transformer pipeline helps model training and inferencing easy by putting together the tasks in sequence and executing them. E.g. putting together preprocessing, model invokation, post-processing during the inferencing as shown below.

Raw_text --> Tokenize (generate token ids) --> MODEL (process the tokens) --> logits --> Prediction (decode the logits into text)

These tasks can be performed individually or you can use the HF pipeline which does them all for you in one go.

### Without Pipeline

In [ ]:
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [ ]:
# tokenize a batch of texts
raw_inputs = ["I love using transformers library!", "This is the worst movie I have ever seen."]
inputs = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors="pt")
print(f"Tokenized inputs: {inputs}")

In [ ]:
# load the pre-trained model
# model = AutoModel.from_pretrained(checkpoint, num_labels=2)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

print(f"Model architecture: {model}")

In [ ]:
# predict sentiment labels
outputs = model(**inputs)
# print(f"Logits: {outputs.last_hidden_state.shape}") # works with AutoModel
# shape: (batch_size, sequence_length, hidden_size)
# (2, 16, 768) for DistilBERT

print(f"Logits Shape: {outputs.logits.shape}")  # works with AutoModelForSequenceClassification
# shape: (batch_size, num_labels)
# (2, 2) for DistilBERT fine-tuned on SST-2
print(f"Logits: {outputs.logits}")

In [ ]:
import torch

# Decoding the predicted labels
predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)

# model.config.id2label(predictions.argmax(dim=-1).item())  # get the label with the highest score
for i, prediction in enumerate(predictions):
    predicted_label = model.config.id2label[prediction.argmax(dim=-1).item()]
    confidence = prediction.max().item()
    print(f"Input Text: {raw_inputs[i]}")
    print(f"Predicted Label: {predicted_label}, Confidence: {confidence:.4f}\n")

### With Pipeline

In [1]:
from transformers import pipeline

classifier = pipeline('sentiment-analysis', framework="pt")
result = classifier(["I love using transformers library!", "I hate waiting in long lines."])
for r in result:
    print(f"Label: {r['label']}, Score: {r['score']:.4f}")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cpu


Label: POSITIVE, Score: 0.9989
Label: NEGATIVE, Score: 0.9969


In [ ]:
from transformers import pipeline

generator = pipeline('text-generation', model='distilgpt2', framework="pt")
output = generator("Once upon a time", max_length=30, num_return_sequences=2)
print(output[0]['generated_text'])

In [ ]:
from transformers import pipeline
unmasker = pipeline('fill-mask', model='bert-base-uncased', framework="pt")
result = unmasker("The capital of France is [MASK].", top_k=3)
for r in result:
    print(f"Token: {r['token_str']}, Score: {r['score']:.4f}")

In [ ]:
from transformers import pipeline
ner = pipeline('ner', grouped_entities=True, framework="pt")
entities = ner("My name is Bibhu and I work at Hugging Face in Dallas.")
for entity in entities:
    print(f"Entity: {entity['entity_group']}, Word: {entity['word']}, Score: {entity['score']:.4f}")

In [ ]:
from transformers import pipeline
summarizer = pipeline("summarization", framework="pt")
text = ("""Plastic surgery is a broad and evolving field of medicine that focuses on the reconstruction, restoration, or alteration of the human body, blending scientific precision with an understanding of aesthetics and human psychology. It is commonly divided into two major branches: reconstructive surgery, which addresses functional impairments caused by trauma, congenital anomalies, burns, or disease, and cosmetic (or aesthetic) surgery, which is primarily concerned with enhancing appearance based on an individual’s personal goals. Procedures such as skin grafts, cleft lip and palate repair, and post-cancer reconstruction play a critical role in restoring not only physical function but also a patient’s confidence and quality of life. On the cosmetic side, surgeries like rhinoplasty, liposuction, breast augmentation, and facelifts have become increasingly common, driven by advancements in surgical techniques, anesthesia, and postoperative care that have improved safety and reduced recovery times. However, plastic surgery is not merely about changing how someone looks; it often intersects with mental health, self-esteem, and societal standards of beauty, raising important ethical and psychological considerations. Surgeons must carefully evaluate a patient’s motivations and expectations, ensuring that procedures are medically appropriate and likely to result in positive outcomes. As technology continues to advance—with innovations such as minimally invasive procedures, laser treatments, and regenerative techniques—plastic surgery continues to expand its capabilities, offering patients more options while emphasizing the importance of informed decision-making, realistic expectations, and holistic well-being.
          """)
result = summarizer(text)
print(result)


2025-12-27 08:27:51.059675: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-27 08:27:51.152687: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-27 08:27:53.266132: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/azureuser/ws/buildllm/.venv/lib/python3.12/site-packa

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

# Model / Model Hub

**Model Hub** is the Central repository for pre-trained and fine-tuned models.

In [ ]:
# How to instantiate and use a Hugging Face Transformer model for text classification
# using the Transformers library.
from transformers import AutoTokenizer, AutoModel

bert_model = AutoModel.from_pretrained("bert-base-uncased")
print(bert_model)

gpt2_model = AutoModel.from_pretrained("gpt2")
print(gpt2_model)

bart_model = AutoModel.from_pretrained("facebook/bart-base")
print(bart_model)

### AutoModel
- Behind the scenes, this API AutoModel.from_pretrained() takes the name of a checkpoint on th hub, downloads & caches the **model config** & **model weights** file.

- It then uses the config class from the **model config** file to create a model class.

- It then creates an instance of the model class using **model config** from the config file. (configs like vocab_size, hidden_dim, n_layers, n_heads, etc...)

- After the model class is instantiated, it loads the **model weights** into it.

- You then have a **pretrained_model** for you to use.

### Change the Default Model Config

In [ ]:
from transformers import BertConfig, BertModel
bert_config = BertConfig.from_pretrained("bert-base-uncased", num_hidden_layers=10)

bert_model = BertModel(bert_config)
print(bert_model)

### Save and Load Model

In [ ]:
bert_model.save_pretrained("./custom-bert-model")
loaded_bert_model = BertModel.from_pretrained("./custom-bert-model")
print(loaded_bert_model)

# Dataset / Dataset Hub

The Dataset Hub is Hugging Face’s central repository for datasets, similar to how the Model Hub hosts models.

- It has 1000s of datasets for NLP, vision, audio, and multimodal.
- public datasets contributed by researchers and organizations.
- It enables you with quick experiments and benchmarking.

The **datasets library** is the Python library to load, process, and manage datasets in your ML workflows.

Example:
```python
from datasets import load_dataset
dataset = load_dataset("imdb")

tokenized = dataset.map(lambda x: tokenizer(x["text"]), batched=True)

large_dataset = load_dataset("wikipedia", streaming=True)

```

# Tokenizer

A tokenizer is a core component in any Transformer-based model. Its job is to convert raw text into a numerical format that models can understand and process efficiently.

In Hugging Face, tokenizers are highly optimized, fast, and fully integrated with the Transformers ecosystem.

- Breaks text into tokens (words, subwords, or characters).
- converts tokens to token_id
- adds special tokenes
- adds padding and truncation

```python
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
text = "Hugging Face makes NLP easy."
tokens = tokenizer(text)
print(tokens)
```
output:
```json
{'input_ids': [464, 3290, 428, 405, 416], 'attention_mask': [1,1,1,1,1]}
```

# Transformer Libraries

The Transformers library is Hugging Face’s core library for working with Transformer-based models. It provides ready-to-use implementations of state-of-the-art NLP (and multimodal) models so you can train, fine-tune, and deploy them **without implementing the complex architecture from scratch.**

**Purpose of the Transformers Library**

  - Standardizes Transformer model implementations (encoder, decoder, encoder-decoder)

  - Provides pre-trained models ready for inference

  - Supports fine-tuning on custom datasets

  - Integrates seamlessly with tokenizers, datasets, Trainer, and pipelines

  - Supports multiple frameworks: PyTorch, TensorFlow, JAX

### Key Components
| Component               | Purpose                                                    | Example                                          |
| ----------------------- | ---------------------------------------------------------- | ------------------------------------------------ |
| `AutoModel`             | Loads model architecture without manually specifying class | `AutoModel.from_pretrained("bert-base-uncased")` |
| `AutoModelForCausalLM`  | Pre-built decoder for language generation                  | GPT-2, LLaMA                                     |
| `AutoModelForSeq2SeqLM` | Encoder-decoder for tasks like translation/summarization   | T5, BART                                         |
| `AutoTokenizer`         | Tokenizes text consistently with the model                 | Converts text → input_ids                        |
| `Trainer`               | Handles fine-tuning, batching, gradient accumulation       | Fine-tune GPT-2 on custom text                   |
| `Pipeline`              | High-level inference for common tasks                      | `pipeline("text-generation")`                    |


In [ ]:
# Example
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")

generator = pipeline("text-generation", model=model, tokenizer=tokenizer)
output = generator("Hugging Face makes NLP", max_length=50)
print(output)

### How fits in the flow

```
Raw Text → Tokenizer → Transformers Model → Output → Post-processing
         ↑                     ↑
       Dataset             Trainer/Pipeline
```
  - **Tokenizer**: converts raw text to input IDs

  - **Model**: applies Transformer architecture for encoding/decoding

  - **Trainer/Pipeline**: handles training/fine-tuning or high-level inference


# Trainer API

The Trainer API is a high-level training abstraction provided by Hugging Face to simplify fine-tuning and training Transformer-based models. It eliminates the need to write low-level training loops while still giving flexibility for advanced customization.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import load_dataset

# Load dataset and tokenizer
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length")

tokenized = dataset.map(tokenize, batched=True)

# Load model
model = AutoModelForCausalLM.from_pretrained("gpt2")

# Data collator
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    overwrite_output_dir=True,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
    logging_dir="./logs",
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
)

# Train
trainer.train()


# Tranformer Application

In this section, we will see some of the transformer based llm applications.

In [3]:
!pip install -U transformers sentencepiece sacremoses

### Text-Classification

In [7]:
from transformers import pipeline
import pandas as pd

# classifier = pipeline('text-classification',
#                       model='distilbert/distilbert-base-uncased-finetuned-sst-2-english')

classifier = pipeline('text-classification',
                      model='SamLowe/roberta-base-go_emotions')

text = "i love the product"
outputs = classifier(text)
df = pd.DataFrame(outputs)
df

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/380 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

Device set to use cpu


,label,score
0,love,0.942236


### Named Entity Recognition

In [11]:
from transformers import pipeline
import pandas as pd

ner_tagger = pipeline('ner', aggregation_strategy="simple")

text = "my name is Bibhu Mishra. I live in Dallas and work for LTI."
outputs = ner_tagger(text)
df = pd.DataFrame(outputs)
df

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496 (https://huggingface.co/dbmdz/bert-large-cased-finetuned-conll03-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


,entity_group,score,word,start,end
0,PER,0.995864,Bibhu Mishra,11,23
1,LOC,0.999092,Dallas,35,41
2,ORG,0.998602,LTI,55,58


### Question and Answer

In [15]:
from transformers import pipeline
import pandas as pd


text = """
Dear Amazon, last week I ordered an Optimus Prime action figure
from your online store in Germany. Unfortunately, when I opened the
package, I discovered to my horror that I had been sent an action
figure of Megatron instead! As a lifelong enemy of the Decepticons,
I hope you can understand my dilemma. To resolve the issue,
I demand an exchange.
"""

reader = pipeline('question-answering')
question = "what was wrong?"
outputs = reader(question=question, context=text)
df = pd.DataFrame([outputs])
df

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5 (https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


,score,start,end,answer
0,0.287915,173,225,I had been sent an action\nfigure of Megatron ...


### Summerization

In [20]:
from transformers import pipeline
import pandas as pd


text = """
Dear Amazon, last week I ordered an Optimus Prime action figure
from your online store in Germany. Unfortunately, when I opened the
package, I discovered to my horror that I had been sent an action
figure of Megatron instead! As a lifelong enemy of the Decepticons,
I hope you can understand my dilemma. To resolve the issue,
I demand an exchange.
"""

summarizer = pipeline('summarization')

output = summarizer(text)
# df = pd.DataFrame([outputs])
# df
output[0]['summary_text']

No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu
Your max_length is set to 142, but your input_length is only 83. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=41)


" Amazon's Optimus Prime action figure was sent to a Decepticons figure instead of Optimus Prime . As a lifelong enemy of the Decepticon, I demand an exchange of the figure from the German retailer . I hope you can understand my dilemma. To resolve the issue, he writes ."

### Text to Music

In [12]:
synth = pipeline("text-to-audio", "facebook/musicgen-small")
text = "a chill song with influences from lofi, chillstep and downtempo"

music = synth(text, forward_params={"do_sample":True})

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu


KeyboardInterrupt: 

In [ ]:
import scipy
scipy.io.wavfile.write("music.wav", rate=music["sampling_rate"], data=music['audio'])

### Text Generation

In [25]:
from transformers import set_seed, pipeline
set_seed(0)
generator = pipeline("text-generation", model="gpt2")
# generator = pipeline("text-generation", model="gpt2-large")

output = generator("there was a lion ")
# print(output[0]['generated_text'])
#
output

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.25G [00:00<?, ?B/s]

KeyboardInterrupt: 

### Translation

In [26]:
from transformers import set_seed, pipeline

set_seed(0)
translator = pipeline("translation_en_to_de")

output= translator("I love this product")
output

No model was supplied, defaulted to google-t5/t5-base and revision a9723ea (https://huggingface.co/google-t5/t5-base).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


[{'translation_text': 'Ich liebe dieses Produkt'}]

### Text to Speech

In [27]:
import transformers
from transformers import pipeline

print(transformers.__version__)

4.57.3


In [30]:
import soundfile as sf
from IPython.display import Audio

text2speech = pipeline("text-to-speech")
text = "OpenAI is a leading AI research organization dedicated to creating safe and powerful artificial intelligence. Its technologies, like GPT models, are transforming how we communicate, learn, and solve complex problems."

synth = text2speech(text)
sf.write("speech.flac", synth["audio"], synth["sampling_rate"])
Audio("speech.flac")



No model was supplied, defaulted to suno/bark-small and revision 1dbd7a1 (https://huggingface.co/suno/bark-small).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:10000 for open-end generation.


KeyboardInterrupt: 